### Persistence in Langgraph

In [3]:
from langgraph.graph import StateGraph, START, END 
from dotenv import load_dotenv 
from langchain_openai import ChatOpenAI 
from typing import TypedDict
from langgraph.checkpoint.memory import InMemorySaver

In [4]:
# loading chatgpt env variable
load_dotenv()

llm = ChatOpenAI()

In [5]:
# BUILDING STATE 

class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str 
    

In [6]:
# generate joke from topic 

def generate_joke(state: JokeState):

    prompt = f"Generate a nide joke on the given topic {state['topic']}"
    response = llm.invoke(prompt).content

    return {'joke':response}

In [7]:
# generate explanation from joke 

def generate_explanation(state:JokeState):

    prompt = f"On the given joke write an explanation of it: {state['joke']}"
    response = llm.invoke(prompt).content 

    return {'explanation':response}


In [8]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke',generate_joke)
graph.add_node('generate_explanation',generate_explanation)

graph.add_edge(START,'generate_joke')
graph.add_edge('generate_joke','generate_explanation')
graph.add_edge('generate_explanation',END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [9]:
# configure the threadid 

config1 = {"configurable":{"thread_id":"1"}}
workflow.invoke({'topic':'pizza'},config=config1)

{'topic': 'pizza',
 'joke': "Why did the pizza go to the therapist? \n\nBecause it had too many toppings and couldn't hold it all together!",
 'explanation': 'This joke is using wordplay to create humor. In this case, the pizza is going to a therapist because it has "too many toppings" which is a play on the idea of someone having too many emotional issues or problems that they need to talk to a therapist about. The punchline, "and couldn\'t hold it all together," is a reference to the fact that a pizza with too many toppings may not stay together and could fall apart, just like how someone with too many issues may feel like they are falling apart. So overall, the joke is playing on the idea of a pizza having toppings as problems and needing therapy to help cope with them.'}

💡 Step-by-step Explanation
1. config1 = {"configurable": {"thread_id": "1"}}

Here, config1 is just a Python dictionary — it holds configuration data for the workflow.

The outer key "configurable" usually means:
👉 "these are runtime settings that can be changed dynamically."

Inside it, "thread_id": "1" assigns a unique identifier (ID) for a specific execution thread or workflow instance.

In many workflow or LLM frameworks (like LangChain, OpenDevin, or LlamaIndex workflows), thread_id helps the system track the context of a particular session or run — similar to a chat session ID.

So this line tells the system:

“Run this workflow in thread 1 — so the system can remember previous steps or maintain continuity for that session.”

In [10]:
# now since we have assigned the thread_id and we can get back to the history and the particular thread or the chat window 

workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': "Why did the pizza go to the therapist? \n\nBecause it had too many toppings and couldn't hold it all together!", 'explanation': 'This joke is using wordplay to create humor. In this case, the pizza is going to a therapist because it has "too many toppings" which is a play on the idea of someone having too many emotional issues or problems that they need to talk to a therapist about. The punchline, "and couldn\'t hold it all together," is a reference to the fact that a pizza with too many toppings may not stay together and could fall apart, just like how someone with too many issues may feel like they are falling apart. So overall, the joke is playing on the idea of a pizza having toppings as problems and needing therapy to help cope with them.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0b8a0a-9b00-61dd-8002-c53c5924822f'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created

In [11]:
# check the history 

list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': "Why did the pizza go to the therapist? \n\nBecause it had too many toppings and couldn't hold it all together!", 'explanation': 'This joke is using wordplay to create humor. In this case, the pizza is going to a therapist because it has "too many toppings" which is a play on the idea of someone having too many emotional issues or problems that they need to talk to a therapist about. The punchline, "and couldn\'t hold it all together," is a reference to the fact that a pizza with too many toppings may not stay together and could fall apart, just like how someone with too many issues may feel like they are falling apart. So overall, the joke is playing on the idea of a pizza having toppings as problems and needing therapy to help cope with them.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0b8a0a-9b00-61dd-8002-c53c5924822f'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, create

In [12]:
# lets try running the another window or chat with another thread 
config2 = {"configurable":{"thread_id":"2"}}
workflow.invoke({"topic":"pasta"},config=config2)

{'topic': 'pasta',
 'joke': 'Why did the spaghetti break up with the linguine? \n\nBecause it was tired of hearing all the penne jokes!',
 'explanation': 'This joke is a play on words using the names of different types of pasta. The spaghetti broke up with the linguine because it was tired of hearing jokes about another type of pasta, penne. The humor comes from the unexpected twist of a pasta relationship breaking up over pasta-related jokes.'}

In [13]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the spaghetti break up with the linguine? \n\nBecause it was tired of hearing all the penne jokes!', 'explanation': 'This joke is a play on words using the names of different types of pasta. The spaghetti broke up with the linguine because it was tired of hearing jokes about another type of pasta, penne. The humor comes from the unexpected twist of a pasta relationship breaking up over pasta-related jokes.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f0b8a0a-e9a7-6dd6-8002-662f4fcfaf66'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2025-11-03T10:34:31.212386+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f0b8a0a-dd4a-643a-8001-1b2a7508d268'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the spaghetti break up with the linguine? \n\nBecause it was tired of hearing all

#### Peristence actually helps with 4 things 

1. Shorttems memory 
2. Fault tolerance 
3. HITL 
4. Time travel:

        Time travel refers to the scenario where we can access any of the checkpoint with the help of the checkpointer_id and reerun the flow 
        and debug it. 

In [18]:
workflow.get_state({"configurable":{"thread_id":"2",'checkpoint_id': '1f0b869f-2684-64b5-8001-52de6c02c481'}})

StateSnapshot(values={}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_id': '1f0b869f-2684-64b5-8001-52de6c02c481'}}, metadata=None, created_at=None, parent_config=None, tasks=(), interrupts=())

In [21]:
workflow.invoke(None, {"configurable":{"thread_id":"2",'checkpoint_id': '1f0b8a0a-dd4a-643a-8001-1b2a7508d268'}})

{'topic': 'pasta',
 'joke': 'Why did the spaghetti break up with the linguine? \n\nBecause it was tired of hearing all the penne jokes!',
 'explanation': 'This joke plays on the similarity in pronunciation between "penne" (a type of pasta) and "pun" (a play on words or joke). The joke implies that the spaghetti broke up with the linguine because it was tired of hearing puns/jokes about penne, another type of pasta. The humor comes from the unexpected twist in the punchline, linking the pasta names to jokes and puns.'}

In [25]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the spaghetti break up with the linguine? \n\nBecause it was tired of hearing all the penne jokes!', 'explanation': 'This joke plays on the similarity in pronunciation between "penne" (a type of pasta) and "pun" (a play on words or joke). The joke implies that the spaghetti broke up with the linguine because it was tired of hearing puns/jokes about penne, another type of pasta. The humor comes from the unexpected twist in the punchline, linking the pasta names to jokes and puns.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f0b8a34-ff5b-62fe-8002-3651fd285e51'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2025-11-03T10:53:20.916761+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f0b8a0a-dd4a-643a-8001-1b2a7508d268'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the spa

In [26]:
# update the state 

workflow.update_state({"configurable":{"thread_id":"2",'checkpoint_id': '1f0b8a0a-dd4a-643a-8001-1b2a7508d268',"checkpoint_ns":""}},{"topic":"samosa"})

{'configurable': {'thread_id': '2',
  'checkpoint_ns': '',
  'checkpoint_id': '1f0b8a84-43a3-6704-8002-8094ff081284'}}

In [31]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the spaghetti break up with the linguine? \n\nBecause it was tired of hearing all the penne jokes!', 'explanation': 'In this joke, the spaghetti breaks up with the linguine because it is tired of hearing jokes that make fun of penne pasta. The joke plays on the similarity in sound between "penne" and "penny," suggesting that the spaghetti is tired of constantly hearing jokes about penne pasta. It\'s a light-hearted play on words that relies on the familiarity of different types of pasta to create humor.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f0b8a8c-2416-6349-8002-edb4760c6c47'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2025-11-03T11:32:20.156704+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f0b8a0a-dd4a-643a-8001-1b2a7508d268'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'samosa'

In [32]:
workflow.invoke(None, {"configurable":{"thread_id":"2"},'checkpoint_id': '1f0b8a84-43a3-6704-8002-8094ff081284'})

{'topic': 'samosa',
 'joke': 'Why did the spaghetti break up with the linguine? \n\nBecause it was tired of hearing all the penne jokes!',
 'explanation': 'This joke is a play on words using the names of different types of pasta. The spaghetti broke up with the linguine because it was tired of hearing jokes about another type of pasta, penne. The humor comes from the unexpected twist of a pasta relationship breaking up over pasta-related jokes.'}

In [34]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the spaghetti break up with the linguine? \n\nBecause it was tired of hearing all the penne jokes!', 'explanation': 'In this joke, the spaghetti breaks up with the linguine because it is tired of hearing jokes that make fun of penne pasta. The joke plays on the similarity in sound between "penne" and "penny," suggesting that the spaghetti is tired of constantly hearing jokes about penne pasta. It\'s a light-hearted play on words that relies on the familiarity of different types of pasta to create humor.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f0b8a8c-2416-6349-8002-edb4760c6c47'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2025-11-03T11:32:20.156704+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f0b8a0a-dd4a-643a-8001-1b2a7508d268'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'samosa'

### Fault tolerance 

In [35]:
from langgraph.graph import StateGraph,START,END 
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time 

In [36]:
# define state 

class CrashState(TypedDict):
    input: str 
    step1: str 
    step2: str 

In [37]:
# define steps 

def step_1(state: CrashState):
    print('Step-1 Execuated')

def step_2(state: CrashState):
    print(" Holding step-2")
    time.sleep(1000)
    return {"step2":"done"}

def step_3(state:CrashState):
    print("Step-3 execuated")
    return {"done":True}

In [38]:

# 3. Build the graph
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [1]:

try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")

▶️ Running graph: Please manually interrupt during Step 2...


NameError: name 'graph' is not defined